In [ ]:
!pip uninstall torchao --yes
!pip install torchao>=0.16.0
!pip show torchao

In [ ]:
!pip install transformers==4.46.3 accelerate peft

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
import json
import peft
from datasets import load_dataset
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
import torch

In [ ]:
path_ = "/content/gdrive/MyDrive/TelegramData/prepared_data"

stack_ = lambda x: os.path.join(path_, x)
train_dataset = load_dataset("json", data_files=stack_("train.json"), split="train")
val_dataset = load_dataset("json", data_files=stack_("val.json"), split="train")
test_dataset = load_dataset("json", data_files=stack_("test.json"), split="train")

In [ ]:
model_name = 'Qwen/Qwen2.5-0.5B'
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

MAX_POST_LEN = 384
MAX_TOTAL_LEN = 512

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def preprocess(ex):
    prompt = ex["input_text"] + "\nКомментарий: "
    target = ex["output_text"] + tokenizer.eos_token

    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_POST_LEN,
    )["input_ids"]

    target_ids = tokenizer(
        target,
        add_special_tokens=False,
        truncation=False,
    )["input_ids"]

    input_ids = prompt_ids + target_ids
    labels = [-100] * len(prompt_ids) + target_ids

    # финальное ограничение длины
    input_ids = input_ids[:MAX_TOTAL_LEN]
    labels = labels[:MAX_TOTAL_LEN]

    # ручной паддинг
    pad_len = MAX_TOTAL_LEN - len(input_ids)
    if pad_len > 0:
        input_ids += [tokenizer.pad_token_id] * pad_len
        labels += [-100] * pad_len

    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": [1] * (MAX_TOTAL_LEN - pad_len) + [0] * pad_len
    }

In [ ]:
save_dir = "/content/gdrive/MyDrive/diploma/lora_weights_ffnn"

In [ ]:
train_dataset = train_dataset.map(preprocess)
val_dataset = val_dataset.map(preprocess)
test_dataset = test_dataset.map(preprocess)

In [ ]:
import gc
def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
cleanup()

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, config)
model.print_trainable_parameters()
# trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093

In [ ]:
training_args = TrainingArguments(
    output_dir=os.path.join(save_dir, model_name),
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)


In [ ]:
trainer.train()

In [ ]:
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

In [ ]:
logs_part1 = trainer.state.log_history.copy()

In [ ]:
post_text = """В октябре световой день в Москве сократится на 2 часа 15 минут.

По данным Московского планетария, в начале месяца солнце будет светить 11 часов 34 минуты, а к концу — только 9 часов 19 минут."""
post_text = test_dataset["input_text"][225]
# cluster_id = 1  # тип реакции
prompt = f"{post_text} \nКомментарий:"

In [ ]:
def generation(model):
  random.seed(42)
  for _ in range(3):
    post_text = test_dataset["input_text"][random.randint(0, 1000)]

    prompt = f"{post_text} \nКомментарий:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=210,
        top_p=0.9,
        no_repeat_ngram_size=3,
        repetition_penalty=1.2,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=False)
    print(generated_text)
    print()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
import random
generation(model)

Видим, что модель снова отвечает "слишком общо" => необходимо еще несколько эпох

In [ ]:
training_args2 = TrainingArguments(
    output_dir=os.path.join(save_dir, model_name),
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=False,
)

trainer = Trainer(
    model=model,
    args=training_args2,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

trainer.train()

In [ ]:
logs_part2 = logs_part1 + trainer.state.log_history

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

In [ ]:
generation(model)

Несмотря на падение loss'а и то, что модель стала отвечать конкретнее, ее ответы не соответствуют смыслу постов.

Продолжим обучение

In [ ]:
training_args3 = TrainingArguments(
    output_dir=os.path.join(save_dir, model_name),
    learning_rate=2e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=False,
)

trainer = Trainer(
    model=model,
    args=training_args3,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

trainer.train()

In [ ]:
generation(model)

Видим, что качество улучшилось, ответы стали ближе по смыслу к посту. Тем не менее - еще не ухватывают достаточно сильно смысл

In [ ]:
logs_part3 = logs_part2 + trainer.state.log_history

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

In [ ]:
training_args4 = TrainingArguments(
    output_dir=os.path.join(save_dir, model_name),
    learning_rate=2e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=False,
)

trainer = Trainer(
    model=model,
    args=training_args4,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

trainer.train()

In [ ]:
import matplotlib.pyplot as plt

logs = trainer.state.log_history
train_loss = [(x["epoch"], x["loss"]) for x in logs if "loss" in x]
val_loss = [(x["epoch"], x["eval_loss"]) for x in logs if "eval_loss" in x]

plt.plot(*zip(*train_loss), label="train")
plt.plot(*zip(*val_loss), label="val")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.savefig("loss_curve.png")
plt.show()